In [3]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)


In [4]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve()
while not (project_root / "data").exists() and project_root != project_root.parent:
    project_root = project_root.parent

bronze_path = project_root / "data" / "bronze" / "bronze_liste_communes_france.csv"
silver_path = project_root / "data" / "silver" / "silver_liste_communes_france.csv"
silver_path.parent.mkdir(parents=True, exist_ok=True)  # ensure /data/silver exists

# Read bronze
df_bronze = pd.read_csv(bronze_path, sep=";", dtype=str, encoding="utf-8")

# Transform -> silver dataframe
df_silver = df_bronze.copy()

# ... your cleaning / renaming / casting on df_silver ...

# Save silver
df_silver.to_csv(silver_path, sep=";", index=False, encoding="utf-8")
print(f"Wrote silver: {silver_path}")


Wrote silver: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/silver/silver_liste_communes_france.csv


In [5]:
df_silver.dtypes

Unnamed: 0                           str
code_insee                           str
nom_standard                         str
nom_sans_pronom                      str
nom_a                                str
nom_de                               str
nom_sans_accent                      str
nom_standard_majuscule               str
typecom                              str
typecom_texte                        str
reg_code                             str
reg_nom                              str
dep_code                             str
dep_nom                              str
canton_code                          str
canton_nom                           str
epci_code                            str
epci_nom                             str
academie_code                        str
academie_nom                         str
code_postal                          str
codes_postaux                        str
zone_emploi                          str
code_insee_centre_zone_emploi        str
code_unite_urbai

In [9]:
string_cols = [
    "code_insee","nom_standard","nom_sans_pronom","nom_a","nom_de","nom_sans_accent",
    "nom_standard_majuscule","typecom","typecom_texte","reg_code","reg_nom","dep_code",
    "dep_nom","canton_code","canton_nom","epci_code","epci_nom","academie_code",
    "academie_nom","code_postal","codes_postaux","zone_emploi","code_insee_centre_zone_emploi",
    "code_unite_urbaine","nom_unite_urbaine","type_commune_unite_urbaine",
    "statut_commune_unite_urbaine","grille_densite_texte","niveau_equipements_services_texte",
    "gentile","url_wikipedia","url_villedereve","extraction_source_url","source_file_name"
]

int_cols = [
    "taille_unite_urbaine","population","superficie_hectare","superficie_km2","densite",
    "altitude_moyenne","altitude_minimale","altitude_maximale","grille_densite",
    "niveau_equipements_services"
]

float_cols = ["latitude_mairie","longitude_mairie","latitude_centre","longitude_centre"]

for c in string_cols:
    if c in df_silver.columns:
        df_silver[c] = df_silver[c].str.strip().astype("string")

for c in int_cols:
    if c in df_silver.columns:
        df_silver[c] = pd.to_numeric(df_silver[c], errors="coerce").astype("Int64")

for c in float_cols:
    if c in df_silver.columns:
        df_silver[c] = pd.to_numeric(df_silver[c].str.replace(",", ".", regex=False), errors="coerce").astype("Float64")

if "ingestion_timestamp" in df_silver.columns:
    df_silver["ingestion_timestamp"] = pd.to_datetime(df_silver["ingestion_timestamp"], errors="coerce")

print(df_silver.dtypes)


Unnamed: 0                                      str
code_insee                                   string
nom_standard                                 string
nom_sans_pronom                              string
nom_a                                        string
nom_de                                       string
nom_sans_accent                              string
nom_standard_majuscule                       string
typecom                                      string
typecom_texte                                string
reg_code                                     string
reg_nom                                      string
dep_code                                     string
dep_nom                                      string
canton_code                                  string
canton_nom                                   string
epci_code                                    string
epci_nom                                     string
academie_code                                string
academie_nom

In [10]:
print(df_silver.columns.tolist())

['Unnamed: 0', 'code_insee', 'nom_standard', 'nom_sans_pronom', 'nom_a', 'nom_de', 'nom_sans_accent', 'nom_standard_majuscule', 'typecom', 'typecom_texte', 'reg_code', 'reg_nom', 'dep_code', 'dep_nom', 'canton_code', 'canton_nom', 'epci_code', 'epci_nom', 'academie_code', 'academie_nom', 'code_postal', 'codes_postaux', 'zone_emploi', 'code_insee_centre_zone_emploi', 'code_unite_urbaine', 'nom_unite_urbaine', 'taille_unite_urbaine', 'type_commune_unite_urbaine', 'statut_commune_unite_urbaine', 'population', 'superficie_hectare', 'superficie_km2', 'densite', 'altitude_moyenne', 'altitude_minimale', 'altitude_maximale', 'latitude_mairie', 'longitude_mairie', 'latitude_centre', 'longitude_centre', 'grille_densite', 'grille_densite_texte', 'niveau_equipements_services', 'niveau_equipements_services_texte', 'gentile', 'url_wikipedia', 'url_villedereve', 'extraction_source_url', 'ingestion_timestamp', 'source_file_name']


In [11]:
# Save silver
df_silver.to_csv(silver_path, sep=";", index=False, encoding="utf-8")
print(f"Wrote silver: {silver_path}")

Wrote silver: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/silver/silver_liste_communes_france.csv


In [15]:
from pathlib import Path
import pandas as pd

# Find repo root (folder containing /data)
project_root = Path.cwd().resolve()
while not (project_root / "data").exists() and project_root != project_root.parent:
    project_root = project_root.parent

b_path = project_root / "data" / "silver" / "silver_burvot_t1_rhone_69.csv"
c_path = project_root / "data" / "silver" / "silver_liste_communes_france.csv"

print("b_path:", b_path)
print("c_path:", c_path)
print("b exists?", b_path.exists())
print("c exists?", c_path.exists())

b = pd.read_csv(b_path, sep=";", dtype=str, encoding="utf-8")
c = pd.read_csv(c_path, sep=";", dtype=str, encoding="utf-8")

b["Code du département"] = b["Code du département"].str.strip()
b["Code de la commune"] = b["Code de la commune"].str.strip()
c["code_insee"] = c["code_insee"].str.strip()

b["code_insee"] = b["Code du département"].str.zfill(2) + b["Code de la commune"].str.zfill(3)

merged = b.merge(
    c,
    on="code_insee",
    how="left",
    validate="many_to_one",
    suffixes=("_burvot", "_communes")
)

print("match ratio:", merged["nom_standard"].notna().mean())


b_path: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/silver/silver_burvot_t1_rhone_69.csv
c_path: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/silver/silver_liste_communes_france.csv
b exists? True
c exists? True
match ratio: 0.9946848899012908


In [16]:
merged.head()

,Code du département,Libellé du département,Code de la circonscription,Libellé de la circonscription,Code de la commune,Libellé de la commune,Code du b.vote,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,N°Panneau.1,Sexe.1,Nom.1,Prénom.1,Voix.1,% Voix/Ins.1,% Voix/Exp.1,N°Panneau.2,Sexe.2,Nom.2,Prénom.2,Voix.2,% Voix/Ins.2,% Voix/Exp.2,N°Panneau.3,Sexe.3,Nom.3,Prénom.3,Voix.3,% Voix/Ins.3,% Voix/Exp.3,N°Panneau.4,Sexe.4,Nom.4,Prénom.4,Voix.4,% Voix/Ins.4,% Voix/Exp.4,N°Panneau.5,Sexe.5,Nom.5,Prénom.5,Voix.5,% Voix/Ins.5,% Voix/Exp.5,N°Panneau.6,Sexe.6,Nom.6,Prénom.6,Voix.6,% Voix/Ins.6,% Voix/Exp.6,N°Panneau.7,Sexe.7,Nom.7,Prénom.7,Voix.7,% Voix/Ins.7,% Voix/Exp.7,N°Panneau.8,Sexe.8,Nom.8,Prénom.8,Voix.8,% Voix/Ins.8,% Voix/Exp.8,N°Panneau.9,Sexe.9,Nom.9,Prénom.9,Voix.9,% Voix/Ins.9,% Voix/Exp.9,N°Panneau.10,Sexe.10,Nom.10,Prénom.10,Voix.10,% Voix/Ins.10,% Voix/Exp.10,N°Panneau.11,Sexe.11,Nom.11,Prénom.11,Voix.11,% Voix/Ins.11,% Voix/Exp.11,extraction_source_url_burvot,ingestion_timestamp_burvot,source_file_name_burvot,code_insee,Unnamed: 0,nom_standard,nom_sans_pronom,nom_a,nom_de,nom_sans_accent,nom_standard_majuscule,typecom,typecom_texte,reg_code,reg_nom,dep_code,dep_nom,canton_code,canton_nom,epci_code,epci_nom,academie_code,academie_nom,code_postal,codes_postaux,zone_emploi,code_insee_centre_zone_emploi,code_unite_urbaine,nom_unite_urbaine,taille_unite_urbaine,type_commune_unite_urbaine,statut_commune_unite_urbaine,population,superficie_hectare,superficie_km2,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre,grille_densite,grille_densite_texte,niveau_equipements_services,niveau_equipements_services_texte,gentile,url_wikipedia,url_villedereve,extraction_source_url_communes,ingestion_timestamp_communes,source_file_name_communes
0,69,Rhône,08,8ème circonscription,001,Affoux,0001,316,60,18.99,256,81.01,4,1.27,1.56,2,0.63,0.78,250,79.11,97.66,1,F,ARTHAUD,Nathalie,0,0,0,2,M,ROUSSEL,Fabien,4,1.27,1.6,3,M,MACRON,Emmanuel,58,18.35,23.2,4,M,LASSALLE,Jean,17,5.38,6.8,5,F,LE PEN,Marine,88,27.85,35.2,6,M,ZEMMOUR,Éric,22,6.96,8.8,7,M,MÉLENCHON,Jean-Luc,27,8.54,10.8,8,F,HIDALGO,Anne,0,0,0,9,M,JADOT,Yannick,6,1.9,2.4,10,F,PÉCRESSE,Valérie,17,5.38,6.8,11,M,POUTOU,Philippe,0,0,0,12,M,DUPONT-AIGNAN,Nicolas,11,3.48,4.4,https://static.data.gouv.fr/resources/election...,2026-03-04T19:37:03.813860,burvot_t1_france_entiere.xlsx,69001,26969,Affoux,Affoux,à Affoux,d'Affoux,affoux,AFFOUX,COM,commune,84,Auvergne-Rhône-Alpes,69,Rhône,6910,Tarare,200040566,CA de l'Ouest Rhodanien,10,Lyon,69170,69170,08430,69243,69000,NaN,0,HORS UNITE URBAINE,H,395,1068,11,37,750,498,960,45.845,4.404,45.844,4.414,6,Rural à habitat dispersé,0,communes non pôle,Affousiens,https://fr.wikipedia.org/wiki/fr:Affoux,https://villedereve.fr/ville/69001-affoux,https://www.data.gouv.fr/api/1/datasets/r/f5df...,2026-03-04 22:08:57.999796,liste_communes_france.csv
1,69,Rhône,09,9ème circonscription,002,Aigueperse,0001,217,58,26.73,159,73.27,5,2.3,3.14,1,0.46,0.63,153,70.51,96.23,1,F,ARTHAUD,Nathalie,1,0.46,0.65,2,M,ROUSSEL,Fabien,2,0.92,1.31,3,M,MACRON,Emmanuel,48,22.12,31.37,4,M,LASSALLE,Jean,8,3.69,5.23,5,F,LE PEN,Marine,48,22.12,31.37,6,M,ZEMMOUR,Éric,9,4.15,5.88,7,M,MÉLENCHON,Jean-Luc,9,4.15,5.88,8,F,HIDALGO,Anne,2,0.92,1.31,9,M,JADOT,Yannick,2,0.92,1.31,10,F,PÉCRESSE,Valérie,15,6.91,9.8,11,M,POUTOU,Philippe,2,0.92,1.31,12,M,DUPONT-AIGNAN,Nicolas,7,3.23,4.58,https://static.data.gouv.fr/resources/election...,2026-03-04T19:37:03.813860,burvot_t1_france_entiere.xlsx,69002,26970,Aigueperse,Aigueperse,à Aigueperse,d'Aigueperse,aigueperse,AIGUEPERSE,COM,commune,84,Auvergne-Rhône-Alpes,69,Rhône,6911,Thizy-les-Bourgs,200067817,CC Saône-Beaujolais,10,Lyon,69790,69790,08434,69264,69000,NaN,0,HORS UNITE URBAINE,H,243,1290,13,19,489,403,610,46.277,4.435,46.28,4.423,7,Rura

In [22]:
merged['N°Panneau.11'].unique()
merged['extraction_source_url_burvot'].unique()


<StringArray>
['https://static.data.gouv.fr/resources/election-presidentielle-des-10-et-24-avril-2022-resultats-definitifs-du-1er-tour/20220414-152612/resultats-par-niveau-burvot-t1-france-entiere.xlsx']
Length: 1, dtype: str

In [4]:
import dtale
dtale.show(merged)

ModuleNotFoundError: No module named 'dtale'